# TAMOC Single Bubble Model (SBM)

Predict bubble rise speed and size evolution using TAMOC's `single_bubble_model`.
All TAMOC API calls are made directly — no wrapper functions.

**Workflow:**
1. Create an ambient `Profile` (world-ocean average or custom)
2. Define a `dbm.FluidParticle` (gas composition)
3. Run `single_bubble_model.Model.simulate()`
4. Extract results with `get_derived_variables()` or use built-in `post_process()` plots
5. Diagnose bubble shape regime using `particle.particle_shape()`

**Configurable parameters** (Cell 2): release depth, bubble diameter, gas composition,
ambient background methane, z-resolution, transfer coefficients.

**References:** Clift, Grace & Weber (1978) for shape regime maps;
Wellek et al. (1966) for ellipsoidal aspect ratio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, contextlib

import gasbubbles
from tamoc import ambient, dbm, seawater, single_bubble_model, dispersed_phases

plt.rcParams["figure.dpi"] = 120

In [ ]:
# ============================================================
# User parameters — edit this cell and rerun cells below
# ============================================================

# Release conditions
z0 = 40.0          # release depth (m)
de = 7.5e-3          # initial bubble diameter (m)
T0 = 273.15 + 12.0   # release temperature (K); set None for ambient temp

# Gas composition and initial mole fractions (must sum to 1.0)
composition = ['methane', 'carbon_dioxide', 'oxygen', 'nitrogen']
mol_frac = np.array([0.93, 0.04, 0.0, 0.03])

# Mass transfer / dissolution controls
K = 1.0             # mass transfer reduction factor (--)
K_T = 1.0           # heat transfer reduction factor (--)
fdis = 1e-8         # dissolved fraction at which particle is "gone"
t_hyd = 0.0         # hydrate film formation time (s); 0 = dirty from start
lag_time = True      # include biodegradation lag time
delta_t = 0.1       # maximum ODE time step (s)

# Ambient: optional background methane concentration (kg/m^3)
# Set to None to skip adding methane to the ambient profile
methane_background_kg_m3 = 3.0e-6

# Post-processing
z_resolution_m = 2.0  # depth grid spacing for regularized output

## Build the ambient profile

`ambient.Profile(None)` loads the world-ocean average T/S data bundled with TAMOC.
`add_computed_gas_concentrations()` adds equilibrium dissolved N₂, O₂, Ar, CO₂.
We can optionally `append()` a constant background methane concentration.

In [ ]:
# Create the ambient profile from TAMOC's built-in world-ocean data
# - Profile(None) loads Levitus et al. (1998) world-ocean average (T, S only)
# - For location-specific data (e.g. North Sea), pass your own CTD data:
#       profile = ambient.Profile(my_data, ztsp=['z','temperature','salinity','pressure'], ...)
#   See examples/ambient/profile_from_ctd.py for how to build custom profiles.
profile = ambient.Profile(None)

# Add equilibrium dissolved gas concentrations (N2, O2, Ar, CO2)
profile.add_computed_gas_concentrations()

# Optionally add background methane
if methane_background_kg_m3 is not None:
    z_pts = profile.interp_ds.coords['z'].values.astype(float)
    ch4_data = np.column_stack([z_pts, np.full_like(z_pts, methane_background_kg_m3)])
    profile.append(ch4_data, ['z', 'methane'], ['m', 'kg/m^3'],
                   comments=['coordinate', 'constant background methane'])

print(f"Profile depth range: {profile.z_min:.0f} – {profile.z_max:.0f} m")
print(f"Chemical variables:  {profile.chem_names}")

## Create the gas particle and run the simulation

`dbm.FluidParticle` with `fp_type=0` creates a gas-phase particle.
`single_bubble_model.Model.simulate()` integrates the bubble trajectory.
`post_process()` shows TAMOC's built-in diagnostic plots.

In [ ]:
# Create the gas bubble particle
gas = dbm.FluidParticle(composition, fp_type=0.)

# Initialize the single bubble model with the ambient profile
sbm = single_bubble_model.Model(profile)

# Run the simulation (suppress TAMOC's verbose print output)
with contextlib.redirect_stdout(open(os.devnull, 'w')):
    sbm.simulate(gas, z0, de, mol_frac, T0,
                 K=K, K_T=K_T, fdis=fdis, t_hyd=t_hyd,
                 lag_time=lag_time, delta_t=delta_t)

# Show TAMOC's built-in post-processing plots
#sbm.post_process(fig=1)

## Extract derived variables and plot rise speed / bubble size vs depth

`get_derived_variables()` returns an array with columns for time, position,
masses, temperature, slip velocity, density, diameter, and transfer coefficients.

In [ ]:
# Extract the derived variables from the simulation
data, var_names, n_chems = sbm.get_derived_variables(track_chems=composition)

# Show the available column names
print("Columns returned by get_derived_variables():")
for i, name in enumerate(var_names):
    print(f"  [{i:2d}] {name}")

# Pull out key arrays for plotting
t = data[:, var_names.index('Simulation time (s)')]
z = data[:, var_names.index('Particle z-coordinate (m)')]
us = data[:, var_names.index('Slip velocity (m/s)')]
de_out = data[:, var_names.index('Particle diameter (m)')]
m_total = data[:, var_names.index('Total mass of all compounds in particle (kg)')]

print(f"\nSimulation: {t[-1]:.1f} s, bubble reached {z[-1]:.1f} m depth")

In [ ]:
# Plot rise speed, diameter, and dissolution vs depth
fig, axs = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

axs[0, 0].plot(us, z, lw=2)
axs[0, 0].invert_yaxis()
axs[0, 0].set(xlabel='Slip velocity (m/s)', ylabel='Depth (m)',
              title='Rise speed vs depth')

axs[0, 1].plot(1e3 * de_out, z, lw=2, c='tab:orange')
axs[0, 1].invert_yaxis()
axs[0, 1].set(xlabel='Equivalent diameter (mm)', ylabel='Depth (m)',
              title='Bubble size vs depth')

axs[1, 0].plot(t, 1e3 * de_out, lw=2, c='tab:green')
axs[1, 0].set(xlabel='Time (s)', ylabel='Equivalent diameter (mm)',
              title='Bubble size vs time')

axs[1, 1].plot(t, 1e6 * m_total, lw=2, c='tab:red')
axs[1, 1].set(xlabel='Time (s)', ylabel='Total mass (mg)',
              title='Dissolution trend')

plt.suptitle(f'SBM: de={1e3*de:.1f} mm, z₀={z0:.0f} m, '
             + '/'.join(f'{c}:{y:.0%}' for c, y in zip(composition, mol_frac)),
             fontsize=10)
plt.show()

# Per-component mass vs depth
fig, ax = plt.subplots(figsize=(7, 5))
for chem in composition:
    col = f'Mass of {chem} in particle (kg)'
    if col in var_names:
        ax.plot(1e6 * data[:, var_names.index(col)], z, label=chem)
ax.invert_yaxis()
ax.set(xlabel='Component mass (mg)', ylabel='Depth (m)',
       title='Component depletion with rise')
ax.legend()
plt.show()

## Bubble shape analysis

TAMOC classifies bubble shape into three regimes (Clift et al. 1978):
- **1 = sphere** (small bubbles, low Eotvos)
- **2 = ellipsoid** (intermediate)
- **3 = spherical cap** (large bubbles, high Eotvos)

`particle.particle_shape(m, T, P, Sa, Ta)` returns `(shape_code, de, rho_p, rho, mu_p, mu, sigma)`.

For ellipsoidal bubbles, a practical aspect ratio estimate uses the **Wellek correlation**:
$$E = \frac{1}{1 + 0.163 \, Eo^{0.757}}$$
where $Eo = \Delta\rho \, g \, d_e^2 / \sigma$ is the Eotvos number.

In [ ]:
# Compute shape code, Eotvos number, and estimated aspect ratio along the trajectory
shape_codes = np.zeros(len(t), dtype=int)
eotvos = np.zeros(len(t))
aspect_ratio = np.zeros(len(t))

g = 9.81
cp = sbm.particle.cp  # heat capacity from the SingleParticle wrapper

for i in range(len(t)):
    # Get masses from the state space (same indexing as get_derived_variables)
    m = sbm.y[i, 3:-1]
    Tp = sbm.y[i, -1] / (np.sum(m) * cp) if np.sum(m) > 0 else 273.15

    # Get ambient conditions at this depth
    Ta, Sa, Pa = profile.get_values(z[i], ['temperature', 'salinity', 'pressure'])

    # TAMOC's shape classification (Clift et al. 1978 regime map)
    shape, d_e, rho_p, rho, mu_p, mu, sigma = gas.particle_shape(m, Tp, Pa, Sa, Ta)

    shape_codes[i] = int(shape)

    # Eotvos number
    if sigma > 0 and d_e > 0:
        eotvos[i] = abs(rho - rho_p) * g * d_e**2 / sigma
    else:
        eotvos[i] = np.nan

    # Aspect ratio estimate
    if shape == 1:
        aspect_ratio[i] = 1.0        # sphere
    elif shape == 2:
        # Wellek et al. (1966) correlation
        aspect_ratio[i] = 1.0 / (1.0 + 0.163 * max(eotvos[i], 0.0)**0.757)
    else:
        aspect_ratio[i] = 0.24       # spherical cap (rough estimate)

# Summary
shape_map = {1: 'sphere', 2: 'ellipsoid', 3: 'spherical_cap'}
unique, counts = np.unique(shape_codes, return_counts=True)
print("Shape regime distribution along trajectory:")
for s, c in zip(unique, counts):
    print(f"  {shape_map.get(s, s):15s}: {c} points")

# Plot
fig, axs = plt.subplots(1, 3, figsize=(14, 5), constrained_layout=True)

axs[0].plot(aspect_ratio, z, lw=2)
axs[0].invert_yaxis()
axs[0].set(xlabel='Aspect ratio (minor/major)', ylabel='Depth (m)',
           title='Estimated bubble flattening')

axs[1].plot(eotvos, z, lw=2, c='tab:purple')
axs[1].invert_yaxis()
axs[1].set(xlabel='Eotvos number', ylabel='Depth (m)',
           title='Shape-driving parameter')

# Color-code the trajectory by shape regime
colors = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:red'}
for s_code, label in shape_map.items():
    mask = shape_codes == s_code
    if np.any(mask):
        axs[2].scatter(us[mask], z[mask], c=colors.get(s_code, 'gray'),
                       s=8, label=label)
axs[2].invert_yaxis()
axs[2].set(xlabel='Slip velocity (m/s)', ylabel='Depth (m)',
           title='Rise speed colored by shape regime')
axs[2].legend()

plt.show()

## Notes and guidance

### What TAMOC already provides
- **Shape classification** via Clift et al. (1978) regime map: Eotvos, Morton, and H-parameter determine sphere / ellipsoid / spherical-cap transitions. Implemented in Fortran (`dbm_phys.f95:particle_shape`).
- **Shape-specific rise velocity**: `us_sphere` (Clift), `us_ellipsoid` (Clift eq 7-4/7-7), `us_spherical_cap` (Clift eq 8-11).
- **Shape-/cleanliness-dependent mass transfer**: clean vs dirty (hydrate-covered) behavior, controlled by `t_hyd`.

### Additional parameters worth exploring
| Parameter | Purpose | How to set |
|---|---|---|
| `t_hyd` | Hydrate film formation time | Use `dispersed_phases.hydrate_formation_time()` for deep releases |
| `K` < 1 | Reduced mass transfer (e.g., surfactant contamination) | Set in parameter cell |
| `K_T` = 0 | Disable heat transfer (assume thermal equilibrium) | Set in parameter cell |
| Ambient currents | Horizontal trajectory (not needed for 1-D rise) | Requires gridded profile |
| Salinity/temperature profiles | Sensitivity to stratification | Build custom `ambient.Profile` |
| Interfacial tension | Custom fluid mixtures | Provide via `dbm` database |

### Aspect ratio: practical options
1. **Quick diagnostic** (used above): Wellek (1966) Eotvos-based estimate for ellipsoids, constant for spheres and caps.
2. **Better physics**: Loth (2008) or Myint et al. (2006) correlations that also depend on Morton number.
3. **Most rigorous**: Experimental calibration for your specific gas/fluid/pressure/temperature range.

### Key references in TAMOC
- Clift, Grace & Weber (1978) — bubble/drop/particle hydrodynamics and regime maps
- Wellek, Agrawal & Skelland (1966) — aspect ratio correlation
- Johnson et al. (1969) — mass transfer for bubbles
- Zheng & Yapa (2000) — spherical-cap rise velocity in ocean bubble modeling
- Jun et al. (2015) — hydrate-film formation timing